# Seven phone models: hierarchical clustering, Ward's method

**Cluster Analysis** · a worked example from the StatNex teaching guide

**The story.** A review site rated seven phone models from 0 to 10 on design, camera, battery, speed, display and value for money. A retailer wants to group them for its shop shelves.

**The question.** Which phones naturally belong together?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-learn`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-learn"

## The table

This cell saves the table `09_multivariate/cluster_analysis__individuals.csv` (7 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Model,Design,Camera,Battery,Speed,Display,Value for money
Aurora X1,9.1,9.3,7.2,9.0,9.4,4.1
Aurora Ultra,9.4,9.6,7.8,9.5,9.6,3.2
Nimbus A5,7.0,7.2,8.1,7.0,7.4,7.6
Nimbus Lite,6.1,5.9,8.4,5.8,6.2,9.0
Kite B2,5.6,5.4,8.9,5.2,5.9,9.3
Kite Pro,7.4,7.8,7.6,7.6,7.9,7.0
Volt G7,8.2,7.5,9.2,9.6,8.8,5.8

These cells save `cluster_merges.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile cluster_merges.py
"""Bookkeeping for hierarchical clustering: who is in which cluster after each merge.

scipy's linkage() returns one row per merge -- (node a, node b, height, size) -- and
calls the cluster formed at step s node n + s. Two rules turn that into the tables
printed here, and neither is in scipy:

* after every merge the clusters are numbered 1, 2, ... by the smallest object number
  each contains, so an object can be followed along its row;
* at every join the branch holding the smaller object number is drawn first, which
  gives the row order of the table (and of the dendrogram).
"""

import numpy as np
import pandas as pd


def merge_trace(merges, n):
    """Cluster numbers after each merge (objects x steps), and the pair merged at each
    step, each side named by its smallest object number (2 x steps)."""
    node = np.arange(n)  # the tree node each object is in now
    memberships, pairs = [], []
    for step, row in enumerate(merges):
        a, b = int(row[0]), int(row[1])
        pairs.append(sorted((np.flatnonzero(node == a).min() + 1,
                             np.flatnonzero(node == b).min() + 1)))
        node[(node == a) | (node == b)] = n + step
        # Numbering clusters in order of first appearance down the objects is
        # numbering them by their smallest object number.
        memberships.append(pd.factorize(node)[0] + 1)
    return np.array(memberships).T, np.array(pairs).T


def leaf_order(merges, n):
    """Objects in drawing order: at each join, the branch with the smaller object first."""
    smallest = list(range(n))
    children = {}
    for step, row in enumerate(merges):
        a, b = int(row[0]), int(row[1])
        smallest.append(min(smallest[a], smallest[b]))
        children[n + step] = (a, b) if smallest[a] < smallest[b] else (b, a)

    def expand(node):
        if node < n:
            return [node]
        first, second = children[node]
        return expand(first) + expand(second)

    return expand(n + len(merges) - 1)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import silhouette_samples
from cluster_merges import leaf_order, merge_trace

pd.set_option("display.width", 120)

## The data

In [ ]:
# Text and blank cells count as missing, and a case missing any variable is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Design", "Camera", "Battery", "Speed", "Display", "Value for money"]
label_column = ["Model"]
numbers = data[variables].apply(pd.to_numeric, errors="coerce")
complete = numbers.notna().all(axis=1)
x = numbers[complete].to_numpy(float)
n, p = x.shape
k = 2
# Each case is named by its label column, or numbered 1, 2, ... among the complete
# cases when there is none.
names = (data.loc[complete, label_column[0]].astype(str).tolist() if label_column
         else [str(i + 1) for i in range(n)])

summary = pd.DataFrame({"n": [n, len(data) - n, len(data)]},
                       index=["Valid cases", "Incomplete cases", "Total"])
summary["share"] = summary["n"] / len(data)
print(summary)

## Descriptive statistics

In [ ]:
# Variance and standard deviation divide by n - 1. A variable with a much larger SD
# than the others weighs more in a Euclidean distance.
describe = pd.DataFrame(
    {"n": n, "mean": x.mean(axis=0), "variance": x.var(axis=0, ddof=1),
     "SD": x.std(axis=0, ddof=1), "min": x.min(axis=0), "max": x.max(axis=0)},
    index=variables,
)
print()
print(describe)
correlation = np.corrcoef(x, rowvar=False)
print()
print(pd.DataFrame(correlation, index=variables, columns=variables).round(4))

## Distances between cases

In [ ]:
labels = names
# Euclidean distance on the variables as they are: the square root of the summed
# squared differences. A variable with a wide spread counts for more.
points = x
# pdist() gives each pair's distance once; squareform() lays them out as a table.
distance = squareform(pdist(points, metric="euclidean"))
print()
print(pd.DataFrame(distance, index=labels, columns=labels).round(4).to_string())

## Merging, closest pair first

In [ ]:
# Ward's method: merge the pair that adds least to the within-cluster sum of squares.
# The height printed for a merge is sqrt(2 x that increase), the scale R's
# hclust(method = "ward.D2") uses, so two single cases merge at their own distance.
# Other programs print the increase itself, or its square (Stata), so compare
# heights only within one program.
method = "ward"
# linkage() returns one row per merge: the two clusters joined, the height (distance)
# at which they join, and the size of the new cluster.
merges = linkage(squareform(distance, checks=False), method=method)
heights = merges[:, 2]
# After each step, clusters are numbered by the smallest object number they hold;
# the rows run in the order the dendrogram draws them (see cluster_merges.py).
# A big jump in height between two steps marks a natural place to cut the tree.
memberships, merged = merge_trace(merges, len(labels))
order = leaf_order(merges, len(labels))
steps = list(range(1, len(labels)))
process = pd.DataFrame(memberships[order], index=[labels[i] for i in order], columns=steps)
process.insert(0, "No.", np.array(order) + 1)
print()
print(pd.DataFrame({"distance": heights, "merged": merged[0], "with": merged[1]}, index=steps))
print(process)

## Cutting the tree into k clusters

In [ ]:
# The membership after the step that leaves k clusters.
assignment = memberships[:, len(labels) - k - 1]
sizes = pd.DataFrame({"size": np.bincount(assignment)[1:]},
                     index=[f"Cluster {c}" for c in range(1, k + 1)])
# Each cluster's mean on every variable: read them to name the clusters.
sizes = sizes.join(pd.DataFrame([x[assignment == c].mean(axis=0) for c in range(1, k + 1)],
                                index=sizes.index, columns=variables))
print()
print(sizes)
for c in range(1, k + 1):
    print(f"Cluster {c}:", ", ".join(labels[i] for i in order if assignment[i] == c))

## Silhouette widths

In [ ]:
# How well the final partition holds together, measured on the distances it was built
# from (k-means: Euclidean between the cases, on the scale k-means used). An object's
# width sets its mean distance a to the rest of its own cluster against its mean
# distance b to the nearest other cluster, s = (b - a) / max(a, b): near 1 it sits
# firmly in its cluster, below 0 the neighbouring cluster is closer. An object alone
# in its cluster counts as 0 (Rousseeuw 1987); scikit-learn's silhouette_samples
# computes them, as R's cluster::silhouette does. Kaufman and Rousseeuw (1990) read
# the overall mean: above 0.70 strong, above 0.50 reasonable, above 0.25 weak (could
# be artificial), else no substantial structure. These are rules of thumb, not tests.
width = silhouette_samples(distance, assignment, metric="precomputed")
group = assignment
silhouette = pd.DataFrame(
    {"size": [*((group == c).sum() for c in range(1, k + 1)), len(group)],
     "mean width": [*(width[group == c].mean() if (group == c).any() else np.nan
                      for c in range(1, k + 1)), width.mean()]},
    index=[*(f"Cluster {c}" for c in range(1, k + 1)), "Total"])
overall = width.mean()
if overall > 0.70:
    rating = "A strong structure"
elif overall > 0.50:
    rating = "A reasonable structure"
elif overall > 0.25:
    rating = "A weak structure (could be artificial)"
else:
    rating = "No substantial structure"
print()
print(silhouette.round(4))
print("Overall:", rating)

## What the result says

Cluster analysis groups cases that are alike. Here it takes the Euclidean distance (the straight-line distance across the six ratings) between seven phones and merges the closest ones step by step with Ward's method.

The merging process starts with three tight merges at distances of 1.14, 1.28 and 1.32: Nimbus Lite with Kite B2, Aurora X1 with Aurora Ultra, and Nimbus A5 with Kite Pro. Volt G7 joins only at 3.95, so it is the loosest fit. With two clusters, a budget pair (value for money 9.15) stands against five better phones (5.54). The mean silhouette width, a score of how well cases sit in their own cluster, is 0.83 for the pair but 0.32 for the five: a weak structure overall (0.47).

So the budget pair is clear, and the five split again into two flagships and a mid-range trio. How many groups to use on the shelves is a business choice.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.